# Cambios en el consumo a nivel local

In [1]:
import pandas as pd

from datetime import datetime, timedelta
from glob import glob
from matplotlib import pyplot as plt

In [2]:
# List of input files
files = glob('data/data_*.csv')

# Concatenate vertically
df = pd.concat([pd.read_csv(f, dtype=str) for f in files], ignore_index=True)

df.head(3)

,hour,zip_code,food,restaurants,hotels,health,airlines,services,goods,leisure,gasoline,total,transaction_online,transaction_physical,date
0,8,08800,168.50,NaN,NaN,NaN,NaN,NaN,168.50,NaN,NaN,168.50,NaN,NaN,2025-05-23
1,11,09839,642.00,NaN,NaN,NaN,NaN,NaN,8252.00,NaN,NaN,8252.00,NaN,NaN,2025-05-23
2,20,11260,791.50,NaN,NaN,7800.61,NaN,NaN,8592.11,NaN,NaN,8592.11,NaN,NaN,2025-05-23


In [7]:
df[['zip_code', 'goods', 'services']]

,zip_code,goods,services
0,08800,168.50,NaN
1,09839,8252.00,NaN
2,11260,8592.11,NaN
3,09839,3641.00,NaN
4,11040,NaN,985216.90
...,...,...,...
347714,10400,926.00,NaN
347715,07050,797.00,NaN
347716,16750,2166.00,NaN
347717,08800,181.50,NaN


---

In [3]:
df = df.query(f'hour in {[str(x) for x in range(0, 24)]}')

In [4]:
df = df.drop(columns=['transaction_online', 'transaction_physical', 'goods', 'services'])

In [5]:
df['date'] = pd.to_datetime(df['date'])

In [6]:
value_cols = [c for c in df.columns if c not in ('datetime', 'date', 'hour', 'zip_code')]

In [7]:
for c in value_cols:
    df[c] = pd.to_numeric(df[c], errors='coerce')

In [8]:
agg_cdmx = df.drop(columns='zip_code').groupby(['date', 'hour']).sum().reset_index()
agg_cdmx['zip_code'] = '00000'

data_hourly = pd.concat([df, agg_cdmx])

In [9]:
def scale(dataframe, grouping_cols, value_cols):
    base = dataframe.copy()[[c for c in dataframe.columns if c not in value_cols]]
    for col in value_cols:
        grouped = dataframe.groupby(grouping_cols)[col]
        min_obs = grouped.transform('min')
        max_obs = grouped.transform('max')
        base[col] = (dataframe.copy()[col] - min_obs) / (max_obs - min_obs)
    return base

In [10]:
data_hourly_per_day = scale(data_hourly, ['zip_code', 'date'], value_cols)
data_hourly_per_day['scaling_reference'] = 'd'
data_hourly_per_day.head(3)

,hour,zip_code,date,food,restaurants,hotels,health,airlines,leisure,gasoline,total,scaling_reference
0,8,08800,2025-05-23,0.051369,NaN,NaN,NaN,NaN,NaN,NaN,0.001093,d
1,11,09839,2025-05-23,0.327585,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,d
2,20,11260,2025-05-23,0.590283,NaN,NaN,1.0,NaN,NaN,NaN,1.000000,d


In [11]:
data_hourly_per_sample = scale(data_hourly, 'zip_code', value_cols)
data_hourly_per_sample['scaling_reference'] = 's'
data_hourly_per_sample.head(3)

,hour,zip_code,date,food,restaurants,hotels,health,airlines,leisure,gasoline,total,scaling_reference
0,8,08800,2025-05-23,0.014924,NaN,NaN,NaN,NaN,NaN,NaN,0.001180,s
1,11,09839,2025-05-23,0.113840,NaN,NaN,NaN,NaN,NaN,NaN,0.056223,s
2,20,11260,2025-05-23,0.170173,NaN,NaN,0.412117,NaN,NaN,NaN,0.372105,s


In [14]:
consumption_hourly = pd.concat([data_hourly_per_day, data_hourly_per_sample], ignore_index=True)
consumption_hourly['datetime'] = consumption_hourly['date'] + consumption_hourly['hour'].apply(lambda x: timedelta(days=int(x)))
consumption_hourly['datetime'] = consumption_hourly['datetime'].apply(lambda x: x.isoformat() + 'Z')
#consumption_hourly['date'] = consumption_hourly['date'].apply(lambda x: x.isoformat().split('T')[0])
#consumption_hourly = consumption_hourly[['scaling_reference', 'zip_code', 'date', 'datetime'] + value_cols]
#consumption_hourly.head(3)

In [15]:
consumption_hourly

,hour,zip_code,date,food,restaurants,hotels,health,airlines,leisure,gasoline,total,scaling_reference,datetime
0,8,08800,2025-05-23,0.051369,NaN,NaN,NaN,NaN,NaN,NaN,0.001093,d,2025-05-31T00:00:00Z
1,11,09839,2025-05-23,0.327585,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,d,2025-06-03T00:00:00Z
2,20,11260,2025-05-23,0.590283,NaN,NaN,1.000000,NaN,NaN,NaN,1.000000,d,2025-06-12T00:00:00Z
3,14,09839,2025-05-23,0.177058,NaN,NaN,NaN,NaN,NaN,NaN,0.438710,d,2025-06-06T00:00:00Z
4,0,11040,2025-05-23,NaN,NaN,0.166567,NaN,NaN,NaN,NaN,0.224713,d,2025-05-23T00:00:00Z
...,...,...,...,...,...,...,...,...,...,...,...,...,...
690659,5,00000,2025-06-01,0.023359,0.013590,0.021979,0.005749,0.000045,0.029937,0.101976,0.050302,s,2025-06-06T00:00:00Z
690660,6,00000,2025-06-01,0.054441,0.013470,0.033318,0.016451,0.000060,0.058751,0.235821,0.081380,s,2025-06-07T00:00:00Z
690661,7,00000,2025-06-01,0.120253,0.012455,0.054987,0.064470,0.001134,0.075174,0.367173,0.108595,s,2025-06-08T00:00:00Z
690662,8,00000,2025-06-01,0.236771,0.040675,0.052124,0.161139,0.001143,0.154921,0.483460,0.183778,s,2025-06-09T00:00:00Z


In [13]:
consumption_daily = scale(data_hourly.drop(columns='hour').groupby(['zip_code', 'date']).sum().reset_index(), 'zip_code', value_cols)
consumption_daily.head(3)

,zip_code,date,food,restaurants,hotels,health,airlines,leisure,gasoline,total
0,00000,2025-05-14,0.128055,0.202062,0.095105,0.576014,0.094349,0.142218,0.502549,0.187193
1,00000,2025-05-15,0.063338,0.427059,0.083771,0.495093,0.085970,0.490239,1.000000,0.295027
2,00000,2025-05-16,0.125254,0.734631,0.507395,0.571783,0.054332,0.412649,0.821816,0.486003


In [14]:
consumption_hourly.round(4).to_csv('cons_h.csv', index=False)
consumption_daily.round(4).to_csv('cons_d.csv', index=False)

In [15]:
cnte = pd.read_csv('cnte.csv')
cnte.head(3)

,date,time,action_type,location,latitude,longitude,notes,source
0,2025-05-15,09:00,Marcha,Ángel de la Independencia,19.423,-99.167,Inicio de la mega marcha hacia el Zócalo,https://www.milenio.com/politica/comunidad/mar...
1,2025-05-15,13:00,Plantón,Zócalo,19.433,-99.132,Instalación de plantón indefinido,https://www.tvazteca.com/aztecanoticias/planto...
2,2025-05-20,08:00,Bloqueo,Auditorio Nacional,19.422,-99.191,Bloqueo simultáneo anunciado,https://www.telediario.mx/comunidad/bloqueos-d...


In [16]:
cnte = cnte.rename(columns={'time': 'hour'})
cnte['hour'] = cnte['hour'].apply(lambda x: int(x.split(':')[0]))

In [17]:
cnte = cnte[[c for c in cnte.columns if c != 'source']]
cnte

,date,hour,action_type,location,latitude,longitude,notes
0,2025-05-15,9,Marcha,Ángel de la Independencia,19.4230,-99.1670,Inicio de la mega marcha hacia el Zócalo
1,2025-05-15,13,Plantón,Zócalo,19.4330,-99.1320,Instalación de plantón indefinido
2,2025-05-20,8,Bloqueo,Auditorio Nacional,19.4220,-99.1910,Bloqueo simultáneo anunciado
3,2025-05-20,8,Bloqueo,Intercambio La Raza,19.4620,-99.1460,Bloqueo simultáneo anunciado
4,2025-05-20,8,Bloqueo,Calz. de Tlalpan & Gral. Anaya,19.3600,-99.1500,Bloqueo simultáneo anunciado
5,2025-05-20,8,Bloqueo,Zaragoza & Blvd. Puerto Aéreo,19.4280,-99.0790,Bloqueo simultáneo anunciado
6,2025-05-22,14,Bloqueo,Paseo de la Reforma (Gral. Prim),19.4310,-99.1530,Cierre de carriles laterales
7,2025-05-23,12,Concentración,Zócalo,19.4330,-99.1320,Concentración prevista
8,2025-05-23,12,Bloqueo,AICM Terminal 1 accesos,19.4360,-99.0730,Bloqueo de accesos vehiculares al aeropuerto
9,2025-06-02,10,Bloqueo,Secretaría de Gobernación (Bucareli & Abraham ...,19.4308,-99.1537,"Bloqueo en Segob; cierre de Bucareli, Abraham ..."


In [18]:
cnte.to_csv('data_cnte.csv', index=False)